In [1]:
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError
import os
import pandas as pd 
import numpy as np 

from metapub import PubMedFetcher 

In [2]:
# Define NCBI error handling decorator with tenacity
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([EutilsNCBIError, EutilsRequestError])
)()

#Create function to retrieve ALL pmids (with NCBI error handling)
@retry_on_communication_error
def Get_list(query):
    fetch = PubMedFetcher()  
    num_of_articles = 500
    start_index = 0
    pmids = []
    while True:
        pmid_batch = fetch.pmids_for_query(query, 
                                      retstart=start_index,
                                      retmax=num_of_articles,
                                      pmc_only= False)
        pmids.extend(pmid_batch)
        start_index = len(pmids)
        if len(pmid_batch) < num_of_articles:
            break     
    return(pmids)

In [3]:
#Read-in query version
with open("PUBMED_query_v1.2", "r") as f:
    file = []
    for line in f:
        file.append(line.replace('\t','').replace('\n','').strip())
query = " ".join(file[1:])

In [5]:
#Run query

a = datetime.now()
START = "2000-01-01"
STOP = "2024-04-01"
start_date_str = START
pmid_list = []
while True: #define periods so that <10,000 are retrieved in the least request calls (best speed)
    if date.fromisoformat(start_date_str) <= date.fromisoformat("2002-07-01"): 
        month_interval = 6
    elif date.fromisoformat(start_date_str) <= date.fromisoformat("2005-11-01"):
        month_interval = 5
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2009-11-01")):
        month_interval = 4
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2011-10-01")):
        month_interval = 3
    elif (date.fromisoformat(start_date_str) <= date.fromisoformat("2023-01-01")):
        month_interval = 2
    else:
        month_interval = 4
    next_start = date.fromisoformat(start_date_str) + relativedelta(months=month_interval)
    end_date = (next_start - relativedelta(days=1))
    end_date_str = end_date.strftime('%Y-%m-%d')
    date_str = f'''(("{start_date_str}"[Date - Publication] : "{end_date_str}"[Date - Publication]) '''
    pmids = Get_list(date_str+query)
    pmids_s = list(set(pmids))
    print(start_date_str,"-", end_date_str,": ", len(pmids_s), f"({len(pmids)})") #checks the number of PMIDs for each quarter(<10,000)
    pmid_list.extend(pmids_s)
    start_date_str = next_start.strftime('%Y-%m-%d')
    if next_start>=date.fromisoformat(STOP):
        end_date_str = STOP
        break
print("Total query duration: ", datetime.now()-a)

2000-01-01 - 2000-06-30 :  7425 (7425)
2000-07-01 - 2000-12-31 :  7391 (7391)
2001-01-01 - 2001-06-30 :  8433 (8433)
2001-07-01 - 2001-12-31 :  7986 (7986)
2002-01-01 - 2002-06-30 :  8711 (8711)
2002-07-01 - 2002-12-31 :  8254 (8254)
2003-01-01 - 2003-05-31 :  7959 (7959)
2003-06-01 - 2003-10-31 :  7780 (7780)
2003-11-01 - 2004-03-31 :  8747 (8747)
2004-04-01 - 2004-08-31 :  8197 (8197)
2004-09-01 - 2005-01-31 :  9520 (9520)
2005-02-01 - 2005-06-30 :  8623 (8623)
2005-07-01 - 2005-11-30 :  9003 (9003)
2005-12-01 - 2006-03-31 :  8532 (8532)
2006-04-01 - 2006-07-31 :  7763 (7763)
2006-08-01 - 2006-11-30 :  7998 (7998)
2006-12-01 - 2007-03-31 :  9724 (9724)
2007-04-01 - 2007-07-31 :  8385 (8385)
2007-08-01 - 2007-11-30 :  8610 (8610)
2007-12-01 - 2008-03-31 :  9417 (9417)
2008-04-01 - 2008-07-31 :  8834 (8834)
2008-08-01 - 2008-11-30 :  8756 (8756)
2008-12-01 - 2009-03-31 :  9725 (9725)
2009-04-01 - 2009-07-31 :  8828 (8828)
2009-08-01 - 2009-11-30 :  8803 (8803)
2009-12-01 - 2010-02-28 :

In [ ]:
pmid_clean = set(pmid_list)
pmid_clean_list = list(pmid_clean)

In [8]:
fetch = PubMedFetcher()  
article = fetch.article_by_pmid(pmid_clean_list[1])
print(article)
print(article.title)
print(article.journal, article.year, article.volume, article.issue)
print(article.authors)
print(article.citation)

<PubMedArticle 35907859> Elkenawy NM; Gomaa OM. Valorization of frying oil waste for biodetergent production using Serratia marcescens N2 and gamma irradiation assisted biorecovery.. Microb Cell Fact. 2022. 21(1):151
Valorization of frying oil waste for biodetergent production using Serratia marcescens N2 and gamma irradiation assisted biorecovery.
Microb Cell Fact 2022 21 1
['Elkenawy NM', 'Gomaa OM']
Elkenawy NM and Gomaa OM. Valorization of frying oil waste for biodetergent production using Serratia marcescens N2 and gamma irradiation assisted biorecovery. Valorization of frying oil waste for biodetergent production using Serratia marcescens N2 and gamma irradiation assisted biorecovery. 2022; 21:151. doi: 10.1186/s12934-022-01877-3


In [ ]:
#save for permanent storage after issues are resolved:
os.makedirs("PMID_lists", exist_ok=True)  # Create if it doesn't exist
date_tag = datetime.now().isoformat()[:10] # Create date tag
np.savetxt('PMID_lists/pmids_'+ date_tag +'.txt', pmid_clean_list, fmt='%s', delimiter=",") #save PMIDs in txt
np.save('PMID_lists/pmids_'+'.npy', pmid_clean_list) #save PMIDS in npy